In [48]:
import os
from pathlib import Path
import sys

import torch
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

In [49]:
from sentence_transformers import CrossEncoder

In [50]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv( override=True )

True

In [51]:
from simonbb.cis438.rag import load_documents, MultiQueryRerankRetriever, ParentDocumentRetriever, HypotheticalQuestionRetriever, build_rag

In [52]:
device = 'cpu'

embedding_model = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5",
    model_kwargs = {'device': device},
    encode_kwargs = {'normalize_embeddings': False}
)

reranker_model = CrossEncoder("BAAI/bge-reranker-large", device=device)

Loading weights: 100%|██████████| 393/393 [00:01<00:00, 233.94it/s]


# Ingestion -> Retrieval -> Generation

## Step 1: Ingestion

Ingestion is the process of splitting documents into **chunks** which are then converted into **embedding** and stored in a **vector store**

In [53]:
docs, success_cnt, page_cnt = load_documents( Path.cwd() / 'cis438.html' )

cis438.html


In [54]:
print( docs[0].page_content[:200] ) # a document object contains 2 attributes: page_content and metadata

Course Description

This course introduces the basics of web-based agentic AI applications, with three main objectives. The primary objective is to introduce the design and implementation of agentic A


In [55]:
print(f"Files: {success_cnt}, Pages: {page_cnt}")

Files: 1, Pages: 1


In [56]:
text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", ". ", " ", ""], chunk_size=600, chunk_overlap=50)

In [57]:
chunks = text_splitter.split_documents(docs)

In [58]:
print(f"Chunks: {len(chunks)}")

Chunks: 15


In [59]:
print( chunks[0].metadata )

{'source': 'c:\\Users\\nizea\\Documents\\CIS438\\rag\\cis438.html'}


In [60]:
vstore = Chroma.from_documents( documents=chunks, embedding=embedding_model )

## Step 2: Retrieval

In [61]:
retriever = vstore.as_retriever(search_kwargs={"k": 4})

In [62]:
query = 'who is the teaching assistant?'

In [63]:
docs = retriever.invoke( query )

In [64]:
def augment_context(input_dict):
    query = input_dict["query"]
    docs = input_dict["context"]
    snippets = []
    sources = {}
    for i, doc in enumerate(docs, start=1):
        sources[i] = {"text": doc.page_content, **doc.metadata}
        snippets.append(f"Snippet {i}: {doc.page_content}")
    return {"query": query, "context": "\n".join(snippets), "sources": sources}

In [65]:
context_aug = augment_context( {"query": query, "context": docs} )
context_aug

{'query': 'who is the teaching assistant?',
 'context': 'Snippet 1: Need help?\n\nTeaching Assistants\n\nJiawei Wang: jwang287@Simon.rochester.edu\n\nTA Office Hours: Friday: 1 -- 2 PM, at CSH 3-327 (aka Fishbowl), southeast end of the hallway with glass doors.\n\nFaculty Office Hours: Wednesday, 12 -- 1 PM, at CSH 3-310\n\nEmail: huaxia.rui@simon.rochester.edu\n\nPhone: 585-275-4278\nSnippet 2: Need help?\n\nTeaching Assistants\n\nJiawei Wang: jwang287@Simon.rochester.edu\n\nTA Office Hours: Friday: 1 -- 2 PM, at CSH 3-327 (aka Fishbowl), southeast end of the hallway with glass doors.\n\nFaculty Office Hours: Wednesday, 12 -- 1 PM, at CSH 3-310\n\nEmail: huaxia.rui@simon.rochester.edu\n\nPhone: 585-275-4278\nSnippet 3: In-Class Exercise: Travel Agent Dec 3 TH Agentic Design Patterns (II) Subgraph Supervisor Reflection LangSmith In-Class Exercise: Coursebot with Tools Dec 4 F Model Contex Protocol (MCP) Building MCP Server Consuming MCP Server Travel Agent App online by 1 PM Dec 8 T In

## Step 3: Generation

In [66]:
llm = ChatOllama( model="llama3", temperature=0.8, num_predict = 256,)

In [18]:
system_prompt = """You are a helpful assistant answering questions based on provided document snippets.

IMPORTANT INSTRUCTIONS:
1. Base your answer ONLY on the provided context snippets below.
2. Synthesize information from multiple snippets if needed.
3. If the snippets contain partial information, provide the best answer you can with what's available.
4. Only say "There wasn't enough information to answer the question" if the snippets truly contain NO relevant information.
5. Always cite your sources using the snippet numbers in parentheses, like (1), (2), etc.

The snippets are provided in this format:
Snippet 1: <text content>
Snippet 2: <text content>
...

{context}"""

In [19]:
prompt = ChatPromptTemplate.from_messages( [ ("system", system_prompt),  ("human", "{query}")  ])

In [20]:
messages = prompt.format_messages( context=context_aug["context"], query=query )

In [21]:
answer = llm.invoke(messages).content
answer

'According to Snippet 1, the Teaching Assistant is Jiawei Wang, who can be contacted at jwang287@Simon.rochester.edu.'

## Step 4: Format Output

In [22]:
def format_output(input_dict):
    answer = input_dict["answer"]
    sources = input_dict.get("sources", {})
    source_docs = []
    for k, v in sources.items():
        doc = Document(
            page_content=v.get('text', ''),
            metadata={key: val for key, val in v.items() if key != 'text'}
        )
        source_docs.append(doc)
    return {"result": answer, "source_docs": source_docs}

In [23]:
response = format_output( {"answer": answer, "sources": context_aug["sources"]} )

In [24]:
response["result"]

'According to Snippet 1, the Teaching Assistant is Jiawei Wang, who can be contacted at jwang287@Simon.rochester.edu.'

In [25]:
citations = response["source_docs"]

In [26]:
from IPython.display import Markdown, display

In [27]:
cites_md = ''
for i, citation in enumerate(citations, 1):
    source = citation.metadata.get('source', 'Unknown')
    page = citation.metadata.get('page', 'N/A')
    cites_md += f"{i}. **Chunk {i}** | Source: {source} | Page: {page} \n {citation.page_content}\n\n"

In [28]:
display(Markdown(cites_md))

1. **Chunk 1** | Source: c:\Users\nizea\Documents\CIS438\rag\cis438.html | Page: N/A 
 Need help?

Teaching Assistants

Jiawei Wang: jwang287@Simon.rochester.edu

TA Office Hours: Friday: 1 -- 2 PM, at CSH 3-327 (aka Fishbowl), southeast end of the hallway with glass doors.

Faculty Office Hours: Wednesday, 12 -- 1 PM, at CSH 3-310

Email: huaxia.rui@simon.rochester.edu

Phone: 585-275-4278

2. **Chunk 2** | Source: c:\Users\nizea\Documents\CIS438\rag\cis438.html | Page: N/A 
 In-Class Exercise: Travel Agent Dec 3 TH Agentic Design Patterns (II) Subgraph Supervisor Reflection LangSmith In-Class Exercise: Coursebot with Tools Dec 4 F Model Contex Protocol (MCP) Building MCP Server Consuming MCP Server Travel Agent App online by 1 PM Dec 8 T In-Person Presentation of Capstone Project Dec 10 (1:40 - 4:40) TH In-Person Presentation of Capstone Project

3. **Chunk 3** | Source: c:\Users\nizea\Documents\CIS438\rag\cis438.html | Page: N/A 
 Quizzes 10% Quizzes are closed-book and are based only on my slides. I usually decide whether to give a quiz at the beginning of each class. Weblog Observatory App 10% Individual-based. Online by Nov 17 at 1 PM. Travel Agent App 10% Individual-based. Online by Dec 4 at 1 PM. Capstone App 50% Individual-based, Online by Dec 8 at 1 PM.  In-person presentation on Dec 8 or Dec 10 Attendance 16% Your attendance score will not be affected unless you miss 8 or more classes. Beyond the exemption of 7 absences, each additional absence will reduce your attendance score by 2 points

4. **Chunk 4** | Source: c:\Users\nizea\Documents\CIS438\rag\cis438.html | Page: N/A 
 Recommended References

There is no textbook. But if you really need some references, here are two relatively recent ones, although they are both somewhat outdated by now.

Learning LangChain, by Mayo Oshin and Nuno Campos, 1st Edition, 2025

AI Agents and Applications, by Roberto Infante, 1st Edition, 2026

Hardware and Software



# Persistence

In [67]:
import chromadb
import sqlite3

In [84]:
store_path = Path.cwd() / 'chroma' 
print(store_path)
client = chromadb.PersistentClient(path=store_path)

c:\Users\nizea\Documents\CIS438\rag\chroma


In [85]:
conn = sqlite3.connect( store_path / "chroma.sqlite3" )
cursor = conn.cursor()
cursor.execute("SELECT collection, id FROM segments WHERE scope = 'VECTOR'")
folder_map = dict(cursor.fetchall())
conn.close()

In [86]:
for collection in client.list_collections():
    client.delete_collection(name=collection.name)

In [97]:
for collection in client.list_collections():
    folder_name = folder_map.get( str(collection.id), "No folder created yet" )
    print( f"Name: {collection.name} | Count: {collection.count()} | Folder: {folder_name}" )

Name: paestum_hypothetical_question | Count: 16 | Folder: No folder created yet
Name: paestum_parent | Count: 0 | Folder: No folder created yet
Name: cis438 | Count: 15 | Folder: No folder created yet
Name: paestum_child | Count: 0 | Folder: No folder created yet
Name: paestum_original_chunks | Count: 4 | Folder: No folder created yet


In [88]:
from chromadb.errors import NotFoundError
def collection_exists(client, collection_name):
    try:
        client.get_collection(name=collection_name)
        return True
    except (ValueError, NotFoundError):
        return False

In [ ]:
collection_name = 'cis438'
if not collection_exists(client, collection_name):
    Chroma.from_documents( documents=chunks, embedding=embedding_model, persist_directory=store_path, collection_name=collection_name )
else:
    print(f'collection {collection_name} exists')

# Parent Document Retriever

In [90]:
kb_name = 'paestum'
kb_path = Path.home() / 'Documents/CIS438/rag/Paestum'
print(kb_path)

C:\Users\nizea\Documents\CIS438\rag\Paestum


In [91]:
retriever = ParentDocumentRetriever(
        persist_directory= store_path,
        embeddings = embedding_model,
        parent_collection = f'{kb_name}_parent',
        child_collection = f'{kb_name}_child',
    )

In [95]:
collection_name = f'{kb_name}_parent'
if not collection_exists(client, collection_name):
    print(f'collection {collection_name} does not exist. Ingesting ...')
    retriever.ingest( docs, source_dir=str(kb_path), parent_chunk_size=1000, child_chunk_size=500 )
else:
    print(f'collection {collection_name} exists')

collection paestum_parent exists


In [38]:
query = 'what is the most famous place to visit?'
retrieved_docs = retriever.invoke( query )

In [39]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nParent {i}:")
    print(doc.id)
    print(doc.metadata)
    print(doc.page_content)

In [40]:
llm = ChatOllama( model="llama3", temperature=0.8, num_predict = 256,)
rag_chain = build_rag( retriever, llm )

In [41]:
response = rag_chain.invoke( query )
response['answer']

"I'm afraid there isn't enough information provided to answer that question. The document snippets don't mention any specific place as the most famous one to visit. Would you like me to help you with something else?"

# Hypothetical Question Retriever

In [42]:
kb_name = 'cis433'
docs, success_cnt, page_cnt = load_documents( Path.cwd() / 'cis433.html' )

cis433.html


In [96]:
collection_name = f'{kb_name}_hypothetical_question'

if not collection_exists(client, collection_name):
    print(f'collection {collection_name} does not exist. Ingesting ...')
    retriever = HypotheticalQuestionRetriever(
        llm = llm,
        persist_directory= store_path,
        embeddings = embedding_model,
        question_collection = f'{kb_name}_hypothetical_question',
        chunk_collection= f'{kb_name}_original_chunks',
        create_collection_if_not_exists = True
    )
    retriever.ingest( docs, source_dir=str(kb_path) )
else:
    print(f'collection {collection_name} exists')
    retriever = HypotheticalQuestionRetriever(
        llm = llm,
        persist_directory= store_path,
        embeddings = embedding_model,
        question_collection = f'{kb_name}_hypothetical_question',
        chunk_collection= f'{kb_name}_original_chunks'
    )

collection paestum_hypothetical_question does not exist. Ingesting ...


In [98]:
query = 'who is the ta?'
retrieved_docs = retriever.invoke( query )

In [99]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Chunk {i}:")
    print(doc.metadata)
    print(doc.page_content)

Chunk 1:
{'chunk_id': 'chunk-28e25d67745335605663489a7cd0725b', 'start_index': 0, 'source': 'c:\\Users\\nizea\\Documents\\CIS438\\rag\\cis438.html', 'source_dir': 'C:\\Users\\nizea\\Documents\\CIS438\\rag\\Paestum'}
Need help?

Teaching Assistants

Jiawei Wang: jwang287@Simon.rochester.edu

TA Office Hours: Friday: 1 -- 2 PM, at CSH 3-327 (aka Fishbowl), southeast end of the hallway with glass doors.

Faculty Office Hours: Wednesday, 12 -- 1 PM, at CSH 3-310

Email: huaxia.rui@simon.rochester.edu

Phone: 585-275-4278
Chunk 2:
{'source_dir': 'C:\\Users\\nizea\\Documents\\CIS438\\rag\\Paestum', 'chunk_id': 'chunk-1af1a960ab72df410519795923815d11', 'source': 'c:\\Users\\nizea\\Documents\\CIS438\\rag\\cis438.html', 'start_index': 0}
Need help?

Teaching Assistants

Jiawei Wang: jwang287@Simon.rochester.edu

TA Office Hours: Friday: 1 -- 2 PM, at CSH 3-327 (aka Fishbowl), southeast end of the hallway with glass doors.

Faculty Office Hours: Wednesday, 12 -- 1 PM, at CSH 3-310

Email: huaxia

In [46]:
rag_chain = build_rag( retriever, llm )

In [47]:
response = rag_chain.invoke( query )
response['answer']

'According to Snippet 1, the Teaching Assistants (TAs) are:\n\n* Adam Li: sli73@simon.rochester.edu\n* Muyang Zhou: mzhou36@simon.rochester.edu\n\nTheir office hours are also specified in Snippet 1: Friday: 2 -- 4 PM at CSH 3-327 (aka Fishbowl).'